# Extract grid data

This notebook will extract plate kinematic data from a plate model and other data from the `source_data` directory, writing the resulting dataset to a CSV file which can then be used to create the time-dependent prospectivity maps in the following notebooks (`02*.ipynb`).

## Notebook setup

These cells set some of the important variables and definitions used throughout the notebook, based on the selected config file.

### Config

In [ ]:
config_file = "config/.run_config.yml"

In [ ]:
from lib.paths import PathConfigManager
pcm = PathConfigManager(config_file, notebook="00c")

# =====================
# Filestructure
# =====================

# Source data paths
plate_model_dir = pcm.PLATE_MODEL_DIR
regions_filepath = pcm.REGIONS_PATH
mantle_data_dir = pcm.MANTLE_DATA_DIR

# Base directories
raster_data_dir = pcm.RASTER_DATA_DIR
points_output_dir = pcm.POINTS_DATA_DIR
grid_output_filename = pcm.GRID_DATA_PATH

pcm.create_directories()

# =====================
# Plate model
# =====================

# Plate model
plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.config["plate_model"]["use_provided_plate_model"]

# Timespan for analysis
min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)


# =====================
# Notebook scope
# =====================

# Gates for determining scope of notebook run (i.e. which features to generate)
use_features = pcm.use_features

# =====================
# Extraction parameters
# =====================

# Buffer distance (degrees) around reference features for sampling unlabelled points
buffer_distance = pcm.config["study_zone_buffer"]

# Resolution of output grids
grid_resolution = pcm.config["grid_resolution"]

# Number of processes to use
n_jobs = pcm.config["n_jobs"]

# Overwrite any existing output files
overwrite = pcm.config["overwrite_output"]

# Control verbosity level of logging output
verbose = pcm.config["verbose"]

### Imports

In [ ]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.assign_regions import assign_regions
from lib.calculate_convergence import run_calculate_convergence
from lib.check_files import (
    check_plate_model,
)
from lib.coregister_combined_point_data import run_coregister_combined_point_data
from lib.coregister_crustal_thickness import run_coregister_crustal_thickness
from lib.coregister_ocean_rasters import (
    extract_subducted_thickness,
    run_coregister_ocean_rasters,
)
from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.pu import generate_grid_points
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness
from lib.sample_mantle import extract_basic_mantle_features

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning

warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [ ]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

In [ ]:
# Seafloor age grid directory
# Filename format 'seafloor_age_{time}Ma.nc'
agegrid_dir = raster_data_dir / "SeafloorAge"

# Seafloor spreading rate directory
# Filename format 'spreading_rate_{time}Ma.nc'
spreadrate_dir = raster_data_dir / "SpreadingRate"

# Seafloor sediment thickness directory
# Filename format 'sediment_thickness_{time}Ma.nc'
sedthick_dir = raster_data_dir / "SedimentThickness"

# Seafloor carbonate sediment thickness directory
# Filename format 'carbonate_thickness_{time}Ma.nc'
carbonate_dir = raster_data_dir / "CarbonateThickness"

# Oceanic crustal CO2 density directory
# Filename format 'crustal_co2_{time}Ma.nc'
co2_dir = raster_data_dir / "CrustalCO2"

# Overriding plate thickness directory
# Filename format 'crustal_thickness_{time}Ma.nc'
crustal_thickness_dir = raster_data_dir / "CrustalThickness"

# Erosion/deposition rate directory
# Filename format 'erosion_deposition_{time}Ma.nc'
erodep_dir = raster_data_dir / "ErosionDeposition"

In [ ]:
# Internal file/directory paths
subduction_data_filename = points_output_dir / "subducting_plate_data.csv"
study_area_dir = points_output_dir / "study_area_polygons"
grid_points_filename = points_output_dir / "grid_points.csv"

# Cumulative grid data set
coregistered_data = None

### Subducting plate data

This cell will extract the subduction kinematics data from the plate model, along with datasets relating to the subducting oceanic plate: seafloor age, sediment and carbonate thickness, etc.
However, if this data has already been extracted by another notebook and `overwrite` has not been set to `True`, then the data will be read from that file instead.

In [ ]:
if use_features('subduction'):
    subduction_data = None
    if overwrite or not subduction_data_filename.is_file():
        subduction_data = run_calculate_convergence(
            nprocs=n_jobs,
            min_time=min(times),
            max_time=max(times),
            plate_reconstruction=plate_model,
            verbose=verbose,
        )
        subduction_data = run_coregister_ocean_rasters(
            nprocs=n_jobs,
            times=times,
            input_data=subduction_data,
            agegrid_dir=agegrid_dir,
            spreadrate_dir=spreadrate_dir,
            plate_reconstruction=plate_model,
            sedthick_dir=sedthick_dir,
            carbonate_dir=carbonate_dir,
            co2_dir=co2_dir,
            verbose=verbose,
        )
        subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
            subduction_data["seafloor_age (Ma)"],
            maxiter=100,
        )
        subduction_data = calculate_water_thickness(data=subduction_data)
        subduction_data = calculate_carbon(subduction_data)
        subduction_data = calculate_slab_flux(subduction_data)
        subduction_data = calculate_slab_dip(subduction_data)
        subduction_data = extract_subducted_thickness(
            subduction_data,
            plate_reconstruction=plate_model,
        )
        subduction_data["sediment_flux (m^2/yr)"] = (
            subduction_data["sediment_thickness (m)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)
        subduction_data["carbon_flux (t/m/yr)"] = (
            subduction_data["total_carbon_density (t/m^2)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)
        subduction_data["water_flux (m^2/yr)"] = (
            subduction_data["total_water_thickness (m)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)

        subduction_data.to_csv(subduction_data_filename, index=False)

### Create study area polygons along subduction zones

Here we define our study area as all points on the overriding plate within a certain distance of the subduction zone (by default, $6 \degree, \approx 660\mathrm{km}$)

In [ ]:
if overwrite or not study_area_dir.is_dir():
    run_create_study_area_polygons(
        nprocs=n_jobs,
        times=times,
        plate_reconstruction=plate_model,
        output_dir=study_area_dir,
        buffer_distance=buffer_distance,
        verbose=verbose,
        return_output=False,
    )

### Generate grid points

The following function generates the grid of points at `grid_resolution`-degree resolution.

In [ ]:
if overwrite or not grid_points_filename.is_file():
    grid_points = generate_grid_points(
        times=times,
        resolution=grid_resolution,
        polygons_dir=study_area_dir,
        plate_reconstruction=plate_model,
        n_jobs=n_jobs,
        verbose=verbose,
    )
    grid_points = grid_points.dropna(subset=["present_lon", "present_lat"])
    grid_points.to_csv(grid_points_filename, index=False)

### Assign subduction data to grid

Here we assign the appropriate values for the subduction-related parameters (kinematics, seafloor age, etc.) to the grid points.

In [ ]:
if use_features('subduction'):
    grid_points = grid_points or pd.read_csv(grid_points_filename)
    subduction_data = subduction_data or pd.read_csv(subduction_data_filename)
    
    coregistered_data = run_coregister_combined_point_data(
        point_data=grid_points,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=verbose,
    )
    
    del grid_points, subduction_data

### Assign crustal thickness data to grid

This cell extracts the overriding plate thickness at each point.

In [ ]:
if use_features('crustal'):
    coregistered_data = coregistered_data or pd.read_csv(grid_points_filename)
    
    coregistered_data = run_coregister_crustal_thickness(
        point_data=coregistered_data,
        input_dir=crustal_thickness_dir,
        n_jobs=n_jobs,
        verbose=verbose,
    )

### Calculate cumulative erosion

Here we calculate the cumulative erosion experienced by each point in the grid since its assigned age time.

In [ ]:
if use_features('erodep'):
    coregistered_data = coregistered_data or pd.read_csv(grid_points_filename)
    
    coregistered_data = calculate_erodep(
        data = coregistered_data,
        input_dir=erodep_dir,
        n_jobs=n_jobs,
        column_name="erosion (m)",
        verbose=verbose,
    )

### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths. Linear interpolation is used to minimise distortion from temporal sparseness of the mantle grids.

In [ ]:
if use_features('mantle'):
    coregistered_data = coregistered_data or pd.read_csv(grid_points_filename)
    
    depths_km = np.arange(100, 2900, 100) # km
    mantle_fields = (
        # "FullTemperature_CG",
        "Pressure",
        "Radial_Velocity",
        # "Temperature_CG",
        "Temperature_Deviation_CG",
        # "Velocity_x",
        # "Velocity_y",
        # "Velocity_z",
        "Viscosity_CG",
        "Velocity_Magnitude",
        "Tangential_Velocity",
        "Radial_Tangential_Ratio",
    )

    coregistered_data = extract_basic_mantle_features(
        points=coregistered_data,
        mantle_dir=mantle_data_dir,
        depths_km=depths_km,
        output_fields=mantle_fields,
    )

### Assign data to regions

To divide the data into individual regions for the later analysis, we use the `regions_filename` defined earlier, if desired.

In [ ]:
if regions_filepath is not None and regions_filepath.is_file():
    points = gpd.GeoSeries.from_xy(
        coregistered_data["present_lon"],
        coregistered_data["present_lat"],
        index=coregistered_data.index,
    )
    coregistered_data["region"] = assign_regions(
        points,
        regions=regions_filepath,
    )
    del points

### Save to file

Finally, we write the dataset to a CSV file.

In [ ]:
coregistered_data.to_csv(grid_output_filename, index=False)